# Notebook 03 — Gráficos interactivos (Plotly)

Visualización interactiva de la dinámica HT vs FT — Premier League 2024/25.
Requiere `plotly`. Los gráficos se renderizan dentro del notebook (JupyterLab).

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

CSV = Path('../data/processed/premier_2024_25_limpio.csv')
df = pd.read_csv(CSV)
print('Shape:', df.shape)
df.head()

Shape: (380, 25)


,date,time,team1,team2,round,ht_g1,ht_g2,ft_g1,ft_g2,league_name,...,imp_ft_menor_ht,imp_mismo_equipo,imp_fecha_invalida,sin_ft,sin_ht,ht_total,ft_total,res_ht,res_ft,remontada
0,2024-08-16,20:00,Manchester United FC,Fulham FC,Matchday 1,0.0,0.0,1,0,English Premier League 2024/25,...,False,False,False,False,False,0.0,1,E,L,False
1,2024-08-17,12:30,Ipswich Town FC,Liverpool FC,Matchday 1,0.0,0.0,0,2,English Premier League 2024/25,...,False,False,False,False,False,0.0,2,E,V,False
2,2024-08-17,15:00,Arsenal FC,Wolverhampton Wanderers FC,Matchday 1,1.0,0.0,2,0,English Premier League 2024/25,...,False,False,False,False,False,1.0,2,L,L,False
3,2024-08-17,15:00,Everton FC,Brighton & Hove Albion FC,Matchday 1,0.0,1.0,0,3,English Premier League 2024/25,...,False,False,False,False,False,1.0,3,V,V,False
4,2024-08-17,15:00,Newcastle United FC,Southampton FC,Matchday 1,1.0,0.0,1,0,English Premier League 2024/25,...,False,False,False,False,False,1.0,1,L,L,False


## 1. Scatter interactivo: goles HT vs goles FT
Cada punto es un partido. Hover muestra equipos, resultado y goles.

In [2]:
d = df.dropna(subset=['ht_total', 'ft_total']).copy()
d['partido'] = d['team1'] + ' vs ' + d['team2']
d['resultado'] = d['res_ft'].map({'L': 'Local', 'E': 'Empate', 'V': 'Visitante'})

fig = px.scatter(
    d, x='ht_total', y='ft_total',
    color='resultado',
    hover_data=['partido', 'date', 'ht_g1', 'ht_g2', 'ft_g1', 'ft_g2'],
    title='Goles medio tiempo (HT) vs tiempo completo (FT) — Premier League 2024/25',
    labels={'ht_total': 'Goles HT (total)', 'ft_total': 'Goles FT (total)'},
    opacity=0.65, width=750, height=550,
)
fig.show()

## 2. Remontadas por equipo (barras interactivas)
% de partidos en los que el equipo perdía en HT y ganó en FT.

In [3]:
d2 = df.dropna(subset=['res_ht', 'res_ft']).copy()

def ganador_ft(r):
    if r['res_ft'] == 'L': return r['team1']
    if r['res_ft'] == 'V': return r['team2']
    return None
def perdedor_ht(r):
    if r['res_ht'] == 'L': return r['team2']
    if r['res_ht'] == 'V': return r['team1']
    return None

d2['ganador_ft'] = d2.apply(ganador_ft, axis=1)
d2['perdedor_ht'] = d2.apply(perdedor_ht, axis=1)
d2['remontada'] = d2['ganador_ft'] == d2['perdedor_ht']

equipos = pd.unique(d2[['team1', 'team2']].values.ravel())
stats = []
for t in equipos:
    pj = d2[(d2['team1'] == t) | (d2['team2'] == t)].shape[0]
    rem = d2[(d2['ganador_ft'] == t) & (d2['remontada'])].shape[0]
    stats.append({'team': t, 'partidos_con_ht': pj, 'remontadas': rem,
                  'pct': round(rem / pj * 100, 1) if pj else 0.0})
s = pd.DataFrame(stats).sort_values('pct', ascending=True)

fig2 = px.bar(
    s, x='pct', y='team', orientation='h',
    hover_data=['partidos_con_ht', 'remontadas'],
    title='% de remontadas por equipo (perdía en HT, ganó en FT)',
    labels={'pct': '% remontadas', 'team': ''},
    color='remontadas', color_continuous_scale='Viridis',
    width=800, height=750,
)
fig2.show()

## 3. Matriz de confusión HT → FT (heatmap interactivo)
Proporción de partidos por fila (resultado HT) que terminaron en cada resultado FT.

In [4]:
d3 = df.dropna(subset=['res_ht', 'res_ft']).copy()
tabla = pd.crosstab(d3['res_ht'], d3['res_ft'], normalize='index')
orden = ['L', 'E', 'V']
etiq = {'L': 'Local', 'E': 'Empate', 'V': 'Visitante'}
tabla = tabla.reindex(index=orden, columns=orden)
tabla.index = [etiq[i] for i in tabla.index]
tabla.columns = [etiq[c] for c in tabla.columns]

fig3 = go.Figure(data=go.Heatmap(
    z=tabla.values, x=list(tabla.columns), y=list(tabla.index),
    colorscale='Blues', text=tabla.values.round(3),
    texttemplate='%{text:.2f}', hovertemplate='HT=%{y} | FT=%{x} | prop=%{z:.3f}<extra></extra>',
))
fig3.update_layout(
    title='Matriz de confusión: resultado HT → FT (proporción por fila)',
    xaxis_title='Resultado FT', yaxis_title='Resultado HT',
    width=600, height=500,
)
fig3.show()

## 4. Distribución de goles totales por jornada
Box plot interactivo de goles FT por cada jornada (Matchday).

In [5]:
d4 = df.dropna(subset=['ft_total']).copy()
d4['jornada'] = d4['round'].str.replace('Matchday ', '', regex=False).astype(int)

fig4 = px.box(
    d4, x='jornada', y='ft_total',
    title='Distribución de goles FT por jornada — Premier League 2024/25',
    labels={'jornada': 'Jornada', 'ft_total': 'Goles FT (total)'},
    width=850, height=450,
)
fig4.show()